In [ ]:
import warnings
warnings.filterwarnings('ignore')

# --- CONFIG ---
CONFIG = {
    "layer2_slm": {
        "enabled": True,
        "model_name": "microsoft/Phi-3-mini-4k-instruct",
        "confidence_threshold": 0.85,
        "max_tokens": 60,
        "temperature": 0.3,
        "escalation_keywords": [
            "policy", "details", "rate", "disbursement",
            "document", "interest", "procedure"
        ],
        "min_response_length": 8,
        "quantization": "4bit",
        "prompt_template": (
            "<|im_start|>system\n"
            "You are a professional BFSI calling agent on a live phone call. "
            "Keep responses short and natural for a phone conversation. "
            "Do not generate customer lines or placeholders. "
            "Do not use placeholders like [insert balance], [insert amount], [insert status] or any bracketed text. "
            "If you do not have the specific information, politely say you will check and get back to the customer. "
            "Never say you are unable to help or apologise for not having information. "
            "Only give your next response as the agent.\n\n"
            "Conversation so far:\n{history}<|im_end|>\n"
            "<|im_start|>user\n{query}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
    }
}

In [ ]:
# --- Install required packages ---
!pip install -q transformers==4.41.2 peft==0.8.2 accelerate==0.27.0 datasets==2.16.1
!pip install -q bitsandbytes>=0.46.1 pandas openpyxl

In [ ]:
# --- Dataset upload & validation ---
from google.colab import files
import json, os, shutil

DATASET_PATH = "/content/dataset.json"

if not os.path.exists(DATASET_PATH):
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    shutil.copy(filename, DATASET_PATH)

with open(DATASET_PATH, 'r') as f:
    alpaca_dataset = json.load(f)
if isinstance(alpaca_dataset, dict) and 'data' in alpaca_dataset:
    alpaca_dataset = alpaca_dataset['data']

required_keys = ['instruction', 'input', 'output']
if not all(key in alpaca_dataset[0] for key in required_keys):
    raise ValueError(f"Dataset format invalid. Required keys: {required_keys}")

In [ ]:
# --- Imports ---
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

RUN_FINETUNING = True  # Set to False to skip training and use a saved model

model_name = CONFIG['layer2_slm']['model_name']

# --- 4-bit Quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# --- Load Base Model ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# --- QLoRA setup ---
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
# --- Dataset formatting ---
dataset = load_dataset('json', data_files=DATASET_PATH)

def format_alpaca(sample):
    return {"text": f"""<|im_start|>system
{sample['instruction']}<|im_end|>
<|im_start|>user
{sample['input']}<|im_end|>
<|im_start|>assistant
{sample['output']}<|im_end|>"""}

dataset = dataset.map(format_alpaca)

def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset['train'].map(
    tokenize_function, batched=True,
    remove_columns=dataset['train'].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

In [ ]:
# --- Training Arguments ---
CHECKPOINT_PATH = "/content/phi3_training_checkpoints"
MODEL_SAVE_PATH  = "/content/phi3_qlora_adapters"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

if RUN_FINETUNING:
    training_args = TrainingArguments(
        output_dir=CHECKPOINT_PATH,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        optim="paged_adamw_8bit",
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    torch.cuda.empty_cache()
    trainer.train()

    model.save_pretrained(MODEL_SAVE_PATH)
    tokenizer.save_pretrained(MODEL_SAVE_PATH)

Step,Training Loss
10,1.786700
20,1.085500
30,0.797000
40,0.705900


In [ ]:
# --------------------------
# Interactive terminal input
# --------------------------

def generate_response(query, model, tokenizer, history):
    prompt_template = CONFIG['layer2_slm']['prompt_template']
    full_prompt = prompt_template.format(history=history, query=query)

    model_inputs = tokenizer([full_prompt], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=CONFIG['layer2_slm']['max_tokens'],
        temperature=CONFIG['layer2_slm']['temperature'],
        do_sample=True if CONFIG['layer2_slm']['temperature'] > 0 else False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the entire generated text
    full_generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

    # Extract only the assistant's response from the full generated text
    # The assistant's response starts after the last "<|im_start|>assistant\n"
    assistant_start_tag = "<|im_start|>assistant\n"
    try:
        # Find the last occurrence of the assistant start tag
        assistant_response_start = full_generated_text.rindex(assistant_start_tag)
        # Extract the text after the tag and before any subsequent <|im_end|> or new <|im_start|>
        response = full_generated_text[assistant_response_start + len(assistant_start_tag):].strip()
        # Remove any subsequent system/user tags or end tags if the model continued beyond expected
        response = response.split("<|im_end|>")[0].strip()
        response = response.split("<|im_start|>")[0].strip()
    except ValueError:
        # If for some reason the tag isn't found, take everything after original prompt
        response = full_generated_text[len(full_prompt):].strip()
        response = response.split("<|im_end|>")[0].strip()
        response = response.split("<|im_start|>")[0].strip()

    # Ensure response is not empty or too short if min_response_length is set
    min_length = CONFIG['layer2_slm']['min_response_length']
    if len(response) < min_length and len(full_generated_text) > len(full_prompt):
        response = full_generated_text[len(full_prompt):].strip().split("<|im_end|>")[0].strip()
        response = response.split("<|im_start|>")[0].strip()

    return response


history = ""  # Keeps multi-turn conversation context

print("Welcome! Type your message and press Enter. Type 'exit' or 'quit' to stop.\n")

while True:
    try:
        user_input = input("You: ")  # Terminal input
    except (KeyboardInterrupt, EOFError):
        print("\nEnding conversation.")
        break

    if user_input.strip().lower() in ["exit", "quit"]:
        print("Ending conversation.")
        break

    # Generate response using your fine-tuned SLM + LoRA adapters
    response = generate_response(user_input, model, tokenizer, history)

    # Print the assistant response
    print("Assistant:", response, "\n")

    # Update conversation history for multi-turn
    history += f"<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>\n"

Welcome! Type your message and press Enter. Type 'exit' or 'quit' to stop.

You: what is my bank balance
Assistant: I'm sorry, but I'm unable to access your bank balance. Could you please call your bank directly? They will be able to help you. 

You: what is my transaction today
Assistant: I'm sorry, but I'm unable to access your transaction details. Could you please call your bank directly? They will be able to help you. 

You: what is emi
Assistant: EMIs stand for Equated Monthly Installments. It's the amount you pay every month towards your loan. It includes both principal and interest. The EMI amount is fixed for the loan tenure and doesn't change. 

You: exit
Ending conversation.


In [ ]:
import json

test_questions = [
    # Exact dataset-style questions
    {"id": "Q1", "question": "What is KYC and why is it required in banking?"},
    {"id": "Q2", "question": "What is the difference between a savings account and a current account?"},
    {"id": "Q3", "question": "What is a credit score and why is it important?"},
    {"id": "Q4", "question": "What is EMI in a loan?"},
    {"id": "Q5", "question": "What is the purpose of insurance?"},

    # New generalization questions
    {"id": "Q6", "question": "Why do banks require identity verification before account opening?"},
    {"id": "Q7", "question": "How is EMI calculated for a personal loan?"},
    {"id": "Q8", "question": "What happens if I miss an EMI payment?"},
    {"id": "Q9", "question": "How can I improve my credit score?"},
    {"id": "Q10", "question": "What is the difference between term insurance and health insurance?"}
]

with open("test_questions.json", "w") as f:
    json.dump(test_questions, f, indent=4)

print("✅ 10 test questions created successfully!")

✅ 10 test questions created successfully!


In [ ]:
with open("test_questions.json", "r") as f:
    questions = json.load(f)

In [ ]:
import pandas as pd
import time

results = []

for item in questions:
    start = time.time()
    response = generate_response(item["question"], model, tokenizer, history="")
    latency = round(time.time() - start, 3)

    results.append({
        "Question_ID": item["id"],
        "Question": item["question"],
        "Phi3_Response": response,
        "Phi3_Latency_s": latency
    })

df_phi3 = pd.DataFrame(results)
df_phi3.to_excel("phi3_results.xlsx", index=False)

print("✅ Phi-3 evaluation complete & saved")

✅ Phi-3 evaluation complete & saved
